In [1]:

import numpy as np
import pandas as pd
import time
import pickle
import warnings
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, roc_curve
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")


# ============================================================
# LOAD DATA
# ============================================================

def load_cleaned_data(filename='data_cleaned.pkl'):
    print("Loading cleaned data...")
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data


# ============================================================
# THRESHOLD OPTIMIZATION
# ============================================================

def find_optimal_threshold(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J = tpr - fpr
    return thresholds[np.argmax(J)]


# ============================================================
# CROSS VALIDATION
# ============================================================

def run_cross_validation(X, y, preprocessor, lr_params, n_splits=10):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        start_time = time.time()

        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)

        model = LogisticRegression(
            C=lr_params['C'],
            penalty=lr_params['penalty'],
            solver=lr_params['solver'],
            max_iter=5000,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_proba = model.predict_proba(X_test)[:, 1]

        threshold = find_optimal_threshold(y_test, y_proba)
        y_pred = (y_proba >= threshold).astype(int)

        results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_test, y_proba),
            "Threshold": threshold,
            "Training Time (s)": time.time() - start_time
        })

    return pd.DataFrame(results)


# ============================================================
# GRID SEARCH
# ============================================================

def grid_search_lr(X, y, preprocessor, param_grid, n_splits=10):

    combinations = list(product(
        param_grid['C'],
        param_grid['penalty'],
        param_grid['solver']
    ))

    all_results = []

    print("\nGRID SEARCH STARTED")
    print("=" * 60)

    for i, params in enumerate(combinations, 1):

        print(f"[{i}/{len(combinations)}] Testing {params}")

        lr_params = {
            "C": params[0],
            "penalty": params[1],
            "solver": params[2]
        }

        cv_df = run_cross_validation(X, y, preprocessor, lr_params, n_splits)

        all_results.append({
            "C": lr_params["C"],
            "penalty": lr_params["penalty"],
            "solver": lr_params["solver"],
            "Accuracy": cv_df["Accuracy"].mean(),
            "Precision": cv_df["Precision"].mean(),
            "Recall": cv_df["Recall"].mean(),
            "F1 Score": cv_df["F1 Score"].mean(),
            "AUC": cv_df["AUC"].mean()
        })

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(
        by=["F1 Score", "Recall"],
        ascending=False
    ).reset_index(drop=True)

    return results_df


# ============================================================
# STABILITY
# ============================================================

def calculate_stability(cv_df):

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]

    summary = []

    for metric in metrics:
        mean_val = cv_df[metric].mean()
        std_val = cv_df[metric].std()
        cv_percent = (std_val / mean_val) * 100 if mean_val != 0 else 0

        if cv_percent < 5:
            status = "SANGAT STABIL"
        elif cv_percent < 10:
            status = "STABIL"
        else:
            status = "KURANG STABIL"

        summary.append({
            "Metric": metric,
            "Mean": mean_val,
            "Std Dev": std_val,
            "CV (%)": cv_percent,
            "Stability": status
        })

    summary_df = pd.DataFrame(summary)

    time_mean = cv_df["Training Time (s)"].mean()
    time_total = cv_df["Training Time (s)"].sum()

    return summary_df, time_mean, time_total


# ============================================================
# MAIN
# ============================================================

def main():

    print("="*70)
    print("LOGISTIC REGRESSION - HYPERPARAMETER TUNING + STABILITY")
    print("="*70)

    data_loaded = load_cleaned_data("data_cleaned.pkl")
    data_cleaned = data_loaded['data_cleaned']
    preprocessor = data_loaded['preprocessor']

    X = data_cleaned.drop(columns=["diagnosis_lanjutan"])
    y = data_cleaned["diagnosis_lanjutan"].values

    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    }

    tuning_results = grid_search_lr(X, y, preprocessor, param_grid)
    tuning_results.to_csv("lr_grid_search_results.csv", index=False)

    best_config = tuning_results.iloc[0]

    lr_params_best = {
        "C": float(best_config["C"]),
        "penalty": best_config["penalty"],
        "solver": best_config["solver"]
    }

    print("\nBEST CONFIGURATION:")
    print(lr_params_best)

    cv_best = run_cross_validation(X, y, preprocessor, lr_params_best)

    print("\nDETAIL 10-FOLD CROSS VALIDATION")
    print(cv_best.to_string(index=False))

    summary_df, time_mean, time_total = calculate_stability(cv_best)

    print("\nRINGKASAN PERFORMA + STABILITAS")
    print(summary_df.to_string(index=False))

    print("\nWAKTU KOMPUTASI:")
    print(f"Rata-rata per fold : {time_mean:.4f} detik")
    print(f"Total waktu        : {time_total:.4f} detik")

    summary_df.to_csv("lr_stability_summary.csv", index=False)

    model_summary = {
        "Model": "Logistic Regression",
        "Accuracy": cv_best["Accuracy"].mean(),
        "Precision": cv_best["Precision"].mean(),
        "Recall": cv_best["Recall"].mean(),
        "F1 Score": cv_best["F1 Score"].mean(),
        "AUC": cv_best["AUC"].mean(),
        "Stability (%)": summary_df["CV (%)"].mean(),
        "Avg Time (s)": time_mean,
        "Total Time (s)": time_total
    }

    model_summary_df = pd.DataFrame([model_summary])

    model_summary_df.to_csv("lr_model_summary.csv", index=False)

    print("\n✅ SELESAI")
    print("File disimpan:")
    print("- lr_grid_search_results.csv")
    print("- lr_stability_summary.csv")


if __name__ == "__main__":
    main()

LOGISTIC REGRESSION - HYPERPARAMETER TUNING + STABILITY
Loading cleaned data...

GRID SEARCH STARTED
[1/4] Testing (0.01, 'l2', 'lbfgs')
[2/4] Testing (0.1, 'l2', 'lbfgs')
[3/4] Testing (1, 'l2', 'lbfgs')
[4/4] Testing (10, 'l2', 'lbfgs')

BEST CONFIGURATION:
{'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}

DETAIL 10-FOLD CROSS VALIDATION
 Fold  Accuracy  Precision   Recall  F1 Score      AUC  Threshold  Training Time (s)
    1  0.956522   0.900000 1.000000  0.947368 0.992063   0.448944           0.041288
    2  0.913043   0.888889 0.888889  0.888889 0.944444   0.452112           0.038499
    3  0.869565   0.875000 0.777778  0.823529 0.892857   0.702393           0.036920
    4  0.913043   0.937500 0.833333  0.882353 0.904762   0.571867           0.032659
    5  0.847826   0.720000 1.000000  0.837209 0.936508   0.096955           0.035941
    6  0.934783   0.900000 0.947368  0.923077 0.927875   0.508619           0.038103
    7  0.891304   0.791667 1.000000  0.883721 0.961014   0.417464